### Exclusão do artefato de reclassificação da EIRELI (natureza_juridica = '2305')

**Entrada:** `data/dataset_modelagem_baixada.parquet` (46.454 linhas) -- checkpoint anterior, produzido por `src/redefinir_alvo.py`, com o alvo `alvo_baixada` já definido (ATIVA vs. BAIXADA por motivo de fracasso real).

**Saída:** `data/dataset_modelagem_final.parquet` (45.876 linhas) -- o mesmo dataset, sem as 578 linhas de EIRELI.

#### 1. Motivo da exclusão

`natureza_juridica == '2305'` (EIRELI) tem 100% de taxa de fracasso, o que causaria separação perfeita na regressão logística. Isso é um artefato de reclassificação legal (extinção da EIRELI em 2021), não um sinal real de risco de negócio -- por isso essas 578 linhas são removidas antes da estimação. O racional completo está no comentário da célula de código abaixo.

#### 2. Carregamento, exclusão e salvamento

Carrega o checkpoint anterior, remove as linhas de EIRELI (seção 1) e salva o resultado como o novo checkpoint para os próximos notebooks.

In [1]:
import pandas as pd

# checkpoint anterior (com alvo_baixada já definido) e checkpoint que este notebook vai produzir
CAMINHO_ENTRADA = "data/dataset_modelagem_baixada.parquet"
CAMINHO_SAIDA = "data/dataset_modelagem_final.parquet"

df = pd.read_parquet(CAMINHO_ENTRADA)
print("Carregado:", df.shape)

Carregado: (46454, 44)


In [2]:
# EIRELI (natureza_juridica='2305') tem 100% de taxa de fracasso (578/578) na base de alvo, o que causaria
# separação perfeita na regressão logística. Mas isso não é risco de negócio real: a EIRELI foi extinta
# pela Lei 14.195/2021, com conversão obrigatória para Sociedade Limitada Unipessoal até meados de 2022
# (o CNPJ passa a registrar sob outro código, tipicamente 2062). No dataset completo (60.000 linhas, todas
# as situações cadastrais), existem 585 estabelecimentos com natureza 2305 e todos são BAIXADA -- nenhum
# ATIVA, nenhum INAPTA -- com data de baixa nunca posterior a 2022. Ou seja: sobreviventes migraram de
# código, e só quem já tinha fechado antes da conversão ficou "congelado" em 2305 -- viés de seleção
# diferencial, não uma relação causal entre ser EIRELI e fracassar. Por isso removemos essas linhas
# antes de estimar o modelo.
mask_eireli = df["natureza_juridica"] == "2305"
print("Linhas EIRELI (2305) a remover:", mask_eireli.sum())

df_final = df[~mask_eireli].copy()
print("Shape apos exclusao:", df_final.shape)

# trava de sanidade: garante que a exclusão removeu exatamente as linhas esperadas, não mais nem menos
assert len(df_final) == 45876, f"Esperado 45876, obtido {len(df_final)}"
print("Assert OK: 45876 linhas")

Linhas EIRELI (2305) a remover: 578
Shape apos exclusao: (45876, 44)
Assert OK: 45876 linhas


In [3]:
# persiste o dataset sem EIRELI como novo checkpoint, para os próximos notebooks partirem daqui
df_final.to_parquet(CAMINHO_SAIDA, index=False)
print(f"Salvo em {CAMINHO_SAIDA}")

Salvo em data/dataset_modelagem_final.parquet


#### 3. Checagem de separação perfeita residual

Depois de remover a EIRELI, confere se sobrou alguma outra categoria de `natureza_juridica` com taxa de fracasso em 0% ou 100% -- sinal de que ainda existiria uma célula problemática para a máxima verossimilhança.

In [4]:
# taxa de fracasso por categoria de natureza_juridica após remover a EIRELI, para saber se ainda
# resta alguma categoria com 0% ou 100% de eventos (o mesmo problema de separação perfeita da EIRELI)
resumo = df_final.groupby("natureza_juridica")["alvo_baixada"].agg(["size", "mean"]).reset_index()
resumo.columns = ["natureza_juridica", "n", "taxa_fracasso"]
resumo["taxa_fracasso_pct"] = (resumo["taxa_fracasso"] * 100).round(1)
resumo = resumo.sort_values("n", ascending=False)

print("=== Taxa de fracasso por natureza_juridica (dataset final) ===")
print(resumo[["natureza_juridica", "n", "taxa_fracasso_pct"]].to_string(index=False))

# Isola categorias com taxa 0% ou 100%: candidatas a nova separação perfeita/quase-separação na regressão.
# Restam 4 com taxa 0% (n=1 ou 2 cada): 2038 (Sociedade de Economia Mista), 2127 (Sociedade em Conta de
# Participação), 2216 (Empresa Domiciliada no Exterior), 3999 (Associação Privada). Ao contrário da EIRELI,
# aqui não há viés de reclassificação -- é só tamanho de célula baixo demais para gerar taxa intermediária
# por acaso. Ainda assim, n tão pequeno com 0% de eventos pode causar quase-separação (coeficiente tendendo
# a -infinito). Tratamento (colapsar em "outras", agrupar por regime/porte, ou aceitar) fica para a próxima etapa.
extremos = resumo[(resumo["taxa_fracasso"] == 0.0) | (resumo["taxa_fracasso"] == 1.0)]
print("\nCategorias com taxa de fracasso 0% ou 100% (separacao perfeita residual):")
print(extremos if len(extremos) else "Nenhuma")

=== Taxa de fracasso por natureza_juridica (dataset final) ===
natureza_juridica     n  taxa_fracasso_pct
             2135 37214               64.8
             2062  8443               34.6
             2054    99               23.2
             2046    98               15.3
             2143    11               27.3
             2240     5               40.0
             3999     2                0.0
             2216     2                0.0
             2038     1                0.0
             2127     1                0.0

Categorias com taxa de fracasso 0% ou 100% (separacao perfeita residual):
  natureza_juridica  n  taxa_fracasso  taxa_fracasso_pct
9              3999  2            0.0                0.0
7              2216  2            0.0                0.0
0              2038  1            0.0                0.0
4              2127  1            0.0                0.0


#### 4. Resultado da checagem -- NÃO está limpo

Mesmo sem a EIRELI, restam 4 categorias de `natureza_juridica` com taxa de fracasso em 0% -- risco de quase-separação, mas por tamanho de célula, não por viés de reclassificação. Este notebook só diagnostica; a decisão de como tratar essas categorias fica para a etapa de especificação do modelo.